In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from datetime import datetime
import warnings

In [2]:
WORKING_DIR = Path("/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project")
INPUT_DIR = WORKING_DIR / "Raw data"
OUT_DIR = WORKING_DIR / "Processed data"
OUT_CSV = OUT_DIR / "NYC_DoF_clean.csv"

In [3]:
MIN_PRICE = 10000
PPSF_MIN = 10
PPSF_MAX = 10000
AGE_MIN = -5
AGE_MAX = 150
LUXURY_PPSF_CUT = 5000

In [4]:
def detect_header_skip(path, lookfor="BOROUGH"):
    """Detect header row containing 'BOROUGH'; fallback = 3"""
    try:
        probe = pd.read_excel(path, header=None, nrows=10)
        for i in range(len(probe)):
            if probe.iloc[i].astype(str).str.contains(lookfor, case=False, na=False).any():
                return max(0, i)
        return 3
    except:
        return 3


def clean_names_min(df):
    """Lowercase column names and replace spaces/hyphens with underscores"""
    df.columns = (df.columns
                  .str.lower()
                  .str.replace(r'[^\w]+', '_', regex=True)
                  .str.strip('_'))
    return df


def parse_sale_date(series):
    """Parse sale dates: Excel serials, ISO, and various date formats"""
    result = pd.Series([pd.NaT] * len(series), index=series.index)
    
    # Try numeric (Excel serial dates)
    numeric_mask = pd.to_numeric(series, errors='coerce').notna()
    if numeric_mask.any():
        nums = pd.to_numeric(series[numeric_mask], errors='coerce')
        excel_mask = (nums > 20000) & (nums < 60000)  # ~1954-2064
        if excel_mask.any():
            result.loc[numeric_mask & excel_mask] = pd.to_datetime(
                nums[excel_mask], origin='1899-12-30', unit='D', errors='coerce'
            )
    
    # Try string parsing for remaining NaT values
    still_nat = result.isna() & series.notna()
    if still_nat.any():
        # Try standard datetime parsing
        result.loc[still_nat] = pd.to_datetime(series[still_nat], errors='coerce')
    
    return result


def normalize_column_names(df):
    """Normalize column names with regex patterns"""
    name_map = {
        r'^(sale_date|saledate)(\W|_)*$': 'sale_date',
        r'^(sale_price)(\W|_)*$': 'sale_price',
        r'^(gross_square_feet|gross_sq_?ft|gross_sf)(\W|_)*$': 'gross_square_feet',
        r'^(land_square_feet|land_sq_?ft|land_sf)(\W|_)*$': 'land_square_feet',
        r'^(apartment_number|apartment|apt)(\W|_)*$': 'apartment_number',
        r'^(building_class_category)(\W|_)*$': 'building_class_category',
        r'^(building_class_at_time_of_sale)(\W|_)*$': 'building_class_at_time_of_sale',
        r'^(tax_class_at_time_of_sale)(\W|_)*$': 'tax_class_at_time_of_sale',
        r'^(building_class_at_present)(\W|_)*$': 'building_class_at_present',
        r'^(tax_class_at_present)(\W|_)*$': 'tax_class_at_present',
        r'^(borough|boro)(\W|_)*$': 'borough',
        r'^(zip_code|zip)(\W|_)*$': 'zip_code',
        r'^(residential_units)(\W|_)*$': 'residential_units',
        r'^(commercial_units)(\W|_)*$': 'commercial_units',
        r'^(total_units)(\W|_)*$': 'total_units',
        r'^(year_built)(\W|_)*$': 'year_built',
        r'^(address)(\W|_)*$': 'address',
        r'^(neighborhood)(\W|_)*$': 'neighborhood',
        r'^(block)(\W|_)*$': 'block',
        r'^(lot)(\W|_)*$': 'lot',
        r'^(ease[_-]?ment)(\W|_)*$': 'easement',
    }
    
    new_cols = df.columns.tolist()
    for pattern, replacement in name_map.items():
        new_cols = [re.sub(pattern, replacement, col, flags=re.IGNORECASE) for col in new_cols]
    
    df.columns = new_cols
    return df


def read_sales_one(path):
    """Read one Excel file robustly"""
    skip = detect_header_skip(path)
    print(f"Reading: {path.name} (skip={skip})")
    
    df = pd.read_excel(path, skiprows=skip, dtype=str)
    df = clean_names_min(df)
    df = normalize_column_names(df)
    df['source_file'] = path.name
    
    return df


def map_borough(series):
    """Map borough codes to names"""
    numeric = pd.to_numeric(series, errors='coerce')
    
    mapping = {
        1: "Manhattan",
        2: "Bronx",
        3: "Brooklyn",
        4: "Queens",
        5: "Staten Island"
    }
    
    result = numeric.map(mapping)
    # For non-numeric, use title case of original
    result = result.fillna(series.astype(str).str.title())
    
    return result

In [5]:
files = list(INPUT_DIR.rglob("*.xls")) + list(INPUT_DIR.rglob("*.xlsx"))
print(f"Total files found: {len(files)}\n")

if len(files) == 0:
    raise ValueError("No Excel files found in input directory")

# Read all files
raw_list = [read_sales_one(f) for f in files]

print("\nBinding rows…")
raw_all = pd.concat(raw_list, ignore_index=True)

# Select wanted columns
want_cols = [
    "borough", "neighborhood", "building_class_category", "tax_class_at_present",
    "block", "lot", "easement", "building_class_at_present", "address",
    "apartment_number", "zip_code", "residential_units", "commercial_units",
    "total_units", "land_square_feet", "gross_square_feet", "year_built",
    "tax_class_at_time_of_sale", "building_class_at_time_of_sale",
    "sale_price", "sale_date", "source_file"
]
raw_all = raw_all[[col for col in want_cols if col in raw_all.columns]]

print("\n================ LOAD SUMMARY ================")
print(f"Rows (raw): {len(raw_all)}")
print(f"Cols (raw): {len(raw_all.columns)}")
print("=============================================\n")

Total files found: 115

Reading: 2013_brooklyn.xls (skip=3)


ImportError: Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.